In [2]:
import numpy as np, random, math, pandas as pd, yfinance as yf, warnings
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import cvxpy as cp

warnings.filterwarnings("ignore")

# Tickers y sectores de 10 empresas del IBEX 35 seleccionadas (2015)
TICKERS = {
    "SAN.MC": "Financiero", "BBVA.MC": "Financiero", "ITX.MC": "Consumo Discrecional",
    "IBE.MC": "Energía Eléctrica", "REP.MC": "Energía (Petróleo y Gas)",
    "TEF.MC": "Telecomunicaciones", "ACS.MC": "Construcción",
    "FER.MC": "Construcción", "GRF.MC": "Salud", "AMS.MC": "Tecnología"
}
tickers = list(TICKERS.keys())

# Descargar precios de cierre ajustados (sin barra de progreso)
prices = yf.download(tickers, start="2015-01-01", end="2015-12-31", auto_adjust=True)["Close"]

# Crear DataFrame con sector de cada empresa
sector_df = pd.DataFrame.from_dict(TICKERS, orient="index", columns=["Sector"])
sector_df.index.name = "Ticker"
sector_df.reset_index(inplace=True)

# Retornos
returns = prices.pct_change().dropna()
mean = returns.mean() * 252
cov = returns.cov() * 252

mu = mean.values
Sigma = cov.values
n = len(mu)
sigma_max = 0.20  # volatilidad máxima (20%)

# Variable de decisión
w = cp.Variable(n)

# Objetivo: maximizar retorno esperado
objective = cp.Maximize(mu @ w)

# Restricciones
constraints = [
    cp.quad_form(w, Sigma) <= sigma_max**2,
    cp.sum(w) == 1,
    w >= 0
]

# Resolución del problema
prob = cp.Problem(objective, constraints)
prob.solve()

# Resultados
w_opt = w.value
ret = (returns.mean() * 252) @ w_opt 
vol = np.sqrt(w_opt @ (returns.cov() * 252).values @ w_opt) 
sharpe = (ret - 0.03) / vol

# === Cálculo de CVaR 95% y Herfindahl
port_daily_returns = returns @ w_opt
sorted_returns = np.sort(port_daily_returns)
cvar_95 = -sorted_returns[:int(0.05 * len(sorted_returns))].mean()
herfindahl_index = np.sum(w_opt**2)

# === Mostrar resultados
print(f"Retorno 2015: {ret*100:.4f}%")
print(f"Volatilidad 2015: {vol*100:.4f}%")
print(f"Sharpe Ratio: {sharpe:.4f}")
print(f"CVaR 95%: {cvar_95:.4f}")
print(f"Índice de Herfindahl (diversificación): {herfindahl_index:.4f}")


[*********************100%***********************]  10 of 10 completed


Retorno 2015: 30.6774%
Volatilidad 2015: 20.0000%
Sharpe Ratio: 1.3839
CVaR 95%: 0.0272
Índice de Herfindahl (diversificación): 0.3869


In [3]:


# Crear DataFrame con los pesos óptimos y los nombres de los activos
pesos_df = pd.DataFrame({
    "Ticker": tickers,
    "Peso (%)": w_opt * 100
})

# Redondear a 2 decimales para que sea más legible
pesos_df["Peso (%)"] = pesos_df["Peso (%)"].round(2)

# Mostrar
print(pesos_df)


    Ticker  Peso (%)
0   SAN.MC      0.00
1  BBVA.MC      5.89
2   ITX.MC      0.00
3   IBE.MC     55.00
4   REP.MC     14.87
5   TEF.MC      0.00
6   ACS.MC     24.25
7   FER.MC      0.00
8   GRF.MC     -0.00
9   AMS.MC      0.00
